# Day 6: Data Quality, Validation & Quarantine DLQ Engine
This notebook runs validation on intentional bad records verifying JSON error manifests routing to quarantine DLQ using PyDeequ.

In [ ]:
!pip install pyspark pyiceberg duckdb splink pydeequ

In [ ]:
import os
os.environ["SPARK_VERSION"] = "3.3"

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, current_date, to_date, expr
from pydeequ.checks import Check, CheckLevel
from pydeequ.verification import VerificationSuite
import pydeequ

spark = SparkSession.builder \
    .appName("Day06_DQ_PyDeequ") \
    .config("spark.jars.packages", "com.amazon.deequ:deequ:2.0.3-spark-3.3") \
    .getOrCreate()

# Sample OMOP Measurement data with intentional errors
data = [
    (1, 3027018, 75.0, "2023-01-01"),   # Valid heart rate
    (2, 3027018, 300.0, "2023-01-02"),  # Invalid heart rate (>250)
    (3, 3027018, 60.0, "2099-12-31"),   # Invalid date (future)
]
columns = ["measurement_id", "measurement_concept_id", "value_as_number", "measurement_date"]
df_raw = spark.createDataFrame(data, columns)
df_raw = df_raw.withColumn("measurement_date", to_date(col("measurement_date")))

print("RAW INCOMING DATA:")
df_raw.show()

# PyDeequ Validation Logic (Dataset level reporting)
check = Check(spark, CheckLevel.Error, "Clinical Plausibility Check")
checkResult = VerificationSuite(spark) \
    .onData(df_raw) \
    .addCheck(
        check.satisfies("measurement_concept_id != 3027018 OR (value_as_number >= 30 AND value_as_number <= 250)", "HR Bounds")
             .satisfies("measurement_date <= current_date()", "No Future Dates")
    ) \
    .run()

print("PYDEEQU VERIFICATION RESULTS:")
checkResult_df = VerificationSuite.checkResultsAsDataFrame(spark, checkResult)
checkResult_df.show(truncate=False)

# Row-level DLQ Routing logic to fulfill strict segregation of bad records
cond_hr = "measurement_concept_id != 3027018 OR (value_as_number >= 30 AND value_as_number <= 250)"
cond_date = "measurement_date <= current_date()"

df_validated = df_raw.withColumn("dq_passed", expr(f"({cond_hr}) AND ({cond_date})"))

df_valid = df_validated.filter("dq_passed = True").drop("dq_passed")
df_dlq = df_validated.filter("dq_passed = False").drop("dq_passed") \
    .withColumn("error_manifest", lit("Failed structural/plausibility rules per PyDeequ definitions"))

print("VALID RECORDS (To Silver):")
df_valid.show()

print("INVALID RECORDS (Routed to DLQ):")
df_dlq.show(truncate=False)
